# Analiza ankiety UX: Gmail vs Outlook
Porównanie interakcji użytkowników na platformach Gmail i Outlook.

## 1. Ładowanie danych z Supabase

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv('..')
url = os.getenv('SUPABASE_URL')
key = os.getenv('SUPABASE_ANON_KEY')

print(f'URL: {url}')
print(f'Key: {key[:20]}...')

In [ ]:
from supabase import create_client

supabase = create_client(url, key)
response = supabase.table('survey_responses').select('*').execute()
data = response.data
print(f'Pobrano {len(data)} odpowiedzi')

## 2. Przegląd danych

In [ ]:
import pandas as pd

df = pd.DataFrame(data)
df['created_at'] = pd.to_datetime(df['created_at'])

print(f'Liczba odpowiedzi: {len(df)}')
print(f'Kolumny: {list(df.columns)}')
print(f'\nPierwsze 5 wierszy:')
df.head()

In [ ]:
# Liczba wypełnionych ocen dla każdego pytania
questions = [f'q{i}' for i in range(1, 9)]
filled = {q: df[f'{q}_gmail'].notna().sum() for q in questions}
pd.Series(filled, name='Liczba odpowiedzi').to_frame()

## 3. Średnie ocen Gmail vs Outlook (wykresy słupkowe)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

titles = {
    'q1': 'Łatwość rozpoczęcia pracy',
    'q2': 'Czytelność nawigacji',
    'q3': 'Wyszukiwanie i filtrowanie',
    'q4': 'Tworzenie i edycja wiadomości',
    'q5': 'Zarządzanie skrzynką',
    'q6': 'Widoczność statusu',
    'q7': 'Wydajność i szybkość',
    'q8': 'Ogólna satysfakcja'
}

means_gmail = [df[f'{q}_gmail'].mean() for q in questions]
means_outlook = [df[f'{q}_outlook'].mean() for q in questions]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Średnie oceny Gmail vs Outlook', fontsize=16, y=1.02)

for i, q in enumerate(questions):
    ax = axes[i // 4][i % 4]
    x = np.arange(2)
    bars = ax.bar(x, [means_gmail[i], means_outlook[i]],
                  color=['#ff6a3d', '#0d8f8d'], width=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(['Gmail', 'Outlook'])
    ax.set_ylim(0, 10)
    ax.set_ylabel('Średnia ocena')
    ax.set_title(f'Q{i+1}: {titles[q]}')
    for bar in bars:
        h = bar.get_height()
        if not np.isnan(h):
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.1,
                    f'{h:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Zbiorczy wykres porównawczy
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(questions))
w = 0.35

bars1 = ax.bar(x - w/2, means_gmail, w, label='Gmail', color='#ff6a3d')
bars2 = ax.bar(x + w/2, means_outlook, w, label='Outlook', color='#0d8f8d')

ax.set_xticks(x)
ax.set_xticklabels([f'Q{i+1}' for i in range(8)])
ax.set_ylim(0, 10)
ax.set_ylabel('Średnia ocena')
ax.set_title('Porównanie średnich ocen: Gmail vs Outlook')
ax.legend()

for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        if not np.isnan(h):
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.1,
                    f'{h:.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

## 4. Rozkład preferencji końcowej

In [ ]:
pref_counts = df['overall_preference'].value_counts()
labels_map = {'gmail': 'Gmail', 'outlook': 'Outlook', 'remis': 'Remis'}
pref_counts.index = [labels_map.get(i, i) for i in pref_counts.index]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Słupkowy
colors = ['#ff6a3d', '#0d8f8d', '#999999']
bars = ax1.bar(pref_counts.index, pref_counts.values, color=colors[:len(pref_counts)])
ax1.set_ylabel('Liczba odpowiedzi')
ax1.set_title('Preferencja końcowa')
for bar in bars:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(int(bar.get_height())), ha='center', va='bottom')

# Kołowy
ax2.pie(pref_counts.values, labels=pref_counts.index, autopct='%1.1f%%',
        colors=colors[:len(pref_counts)], startangle=90)
ax2.set_title('Rozkład preferencji')

plt.tight_layout()
plt.show()

## 5. Liczba odpowiedzi w czasie

In [ ]:
df_sorted = df.sort_values('created_at')
df_sorted['cumcount'] = range(1, len(df_sorted) + 1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_sorted['created_at'], df_sorted['cumcount'],
        marker='o', color='#0d8f8d', linewidth=2)
ax.set_xlabel('Data')
ax.set_ylabel('Liczba odpowiedzi (skumulowana)')
ax.set_title('Przyrost odpowiedzi w czasie')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Łączna liczba odpowiedzi: {len(df)}')

## 6. Komentarze uczestników

In [ ]:
comments = df[df['overall_comment'].notna() & (df['overall_comment'] != '')][['created_at', 'participant_name', 'overall_comment']]
if len(comments):
    display(comments)
else:
    print('Brak komentarzy.')